# MWM — long-history retrain (Colab)

Runs the end-to-end long-history retrain (O1–O3 + precision/recall) on Colab GPU,
**one instrument per session** at **60 epochs**.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is enough).
Set `INSTRUMENT` in cell 2, run top to bottom, then repeat the session for the next one.

Why 60 and not 200: the 2026-07-16 run bottomed out at val=0.0121 by epoch 48 and only
drifted upward through epoch 84 (train kept falling to 0.0031 — plain overfitting).
Epochs past ~60 cost GPU hours and give back a worse model.

Data & checkpoints persist to Google Drive, so a disconnect resumes instead of restarting.
The Dukascopy pull is cached per-chunk to Drive: slow once, instant on later sessions.

Honest per-instrument start dates (gated by macro history) are used automatically:
gold 2008 (GVZ), EUR/USD 2004, USD/JPY 2006.

In [ ]:
# 1) Check GPU + RAM
import torch, psutil, subprocess
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('RAM GB:', round(psutil.virtual_memory().total/2**30, 1))
# If CUDA is False: Runtime -> Change runtime type -> GPU, then rerun.

In [ ]:
# 2) Mount Drive + pick THIS session's instrument
from google.colab import drive
drive.mount('/content/drive')
import os

# >>> The only knobs. One instrument per session: run 'gold', then 'eurusd', then 'usdjpy'. <<<
INSTRUMENT = 'gold'          # 'gold' | 'eurusd' | 'usdjpy'
EPOCHS     = 60              # val bottoms ~epoch 48; 60 leaves headroom without overfitting

assert INSTRUMENT in ('gold', 'eurusd', 'usdjpy'), INSTRUMENT
PERSIST = '/content/drive/MyDrive/MWM_run'   # data cache + checkpoints live here
os.makedirs(PERSIST, exist_ok=True)
print(f'instrument={INSTRUMENT}  epochs={EPOCHS}')
print('persisting to', PERSIST)

## 3) Get the repo into Colab
The project isn't a git repo, so upload it.

**Upload `MWM_slim.zip` (~4 MB)** from your Downloads folder to `/content/` via the Files panel.
Wait for the upload to finish before running the next cell -- Colab shows the filename as soon
as the transfer *starts*, and running early hands `zipfile` a half-written file (`BadZipFile`).

Do not upload the old `MWM.zip`: it is 267 MB, 234 MB of which is `.pt` checkpoints Colab never
reads (training writes fresh ones to Drive). Rebuild the slim zip any time with
`scratchpad/make_slim_zip.py`.


In [ ]:
# 3) Unzip the uploaded repo to /content/MWM
# Upload MWM_slim.zip (~4 MB) via the Files panel. The old MWM.zip was 267 MB --
# 234 MB of it .pt checkpoints Colab never reads. Large uploads truncate and then
# surface as BadZipFile. This cell validates every candidate zip rather than
# trusting the first glob hit, and names the broken one instead of dying on it.
import os, zipfile, glob, shutil

if not os.path.isdir('/content/MWM'):
    cands = sorted(glob.glob('/content/*.zip'))
    assert cands, 'Upload MWM_slim.zip to /content/ first (Files panel).'

    good = None
    for c in cands:
        mb = os.path.getsize(c) / 1048576
        try:
            with zipfile.ZipFile(c) as z:
                if any(n.endswith('data/long_history.py') for n in z.namelist()):
                    good = c
                    print(f'OK    {c}  ({mb:.1f} MB)')
                    break
                print(f'skip  {c}  ({mb:.1f} MB) - valid zip, but no MWM inside')
        except zipfile.BadZipFile:
            print(f'BAD   {c}  ({mb:.1f} MB) - truncated or not a zip; delete and re-upload')

    assert good, ('No usable MWM zip in /content. A file marked BAD above means the '
                  'upload did not finish: delete it in the Files panel and re-upload '
                  'MWM_slim.zip (~4 MB).')

    shutil.rmtree('/content/MWM_extract', ignore_errors=True)
    with zipfile.ZipFile(good) as z:
        z.extractall('/content/MWM_extract')
    hit = glob.glob('/content/MWM_extract/**/data/long_history.py', recursive=True)
    assert hit, 'Could not find data/long_history.py in the zip.'
    os.rename(os.path.dirname(os.path.dirname(hit[0])), '/content/MWM')

os.chdir('/content/MWM')
print('cwd:', os.getcwd())
print(sorted(os.listdir('.'))[:12])


In [ ]:
# 4b) Guard against a STALE zip.
# Cell 3 skips extraction when /content/MWM already exists, so an old upload keeps
# an old training/train_o1.py. On 2026-07-16 that silently let the final epoch
# overwrite best_model.pt (gold: best 0.0117@26 clobbered by 0.0124@60).
# This cell repairs the checkpoint logic in place and refuses to continue if it can't.
import pathlib, sys

TRAIN_PY = pathlib.Path('/content/MWM/training/train_o1.py')
src = TRAIN_PY.read_text(encoding='utf-8', errors='surrogateescape')

if 'last_model.pt' in src:
    print('train_o1.py already current -- no patch needed')
else:
    L = src.split(chr(10))
    GUARD = '        if ckpt_score < best_val_loss or is_last_epoch:'
    INNER = '            if ckpt_score < best_val_loss:'
    SAVE = '            torch.save(ckpt_payload, f"{ckpt_dir}/best_model.pt")'
    PRINT = '            print(f"  Checkpoint saved ['

    gi = [i for i, l in enumerate(L) if l == GUARD]
    si = [i for i, l in enumerate(L) if l == SAVE]
    assert len(gi) == 1 and len(si) == 1, f'anchors not unique: {gi} {si} -- re-upload the zip'
    gi, si = gi[0], si[0]
    assert L[gi + 1] == INNER, f'unexpected line: {L[gi+1]!r} -- re-upload the zip'
    ei = next(i for i in range(si, si + 8) if L[i].startswith(PRINT))

    L[gi] = '        is_best = ckpt_score < best_val_loss' + chr(10) + '        if is_best or is_last_epoch:'
    L[gi + 1] = '            if is_best:'
    L[si:ei + 1] = [
        '            if is_best:',
        '                torch.save(ckpt_payload, f"{ckpt_dir}/best_model.pt")',
        '                print(f"  Checkpoint saved [{ckpt_score:.4f}]")',
        '            if is_last_epoch:',
        '                torch.save(ckpt_payload, f"{ckpt_dir}/last_model.pt")',
        '                print(f"  Final epoch saved [last_model.pt={ckpt_score:.4f}]")',
    ]
    out = chr(10).join(L)
    compile(out, str(TRAIN_PY), 'exec')       # never write anything unparseable
    TRAIN_PY.write_text(out, encoding='utf-8', errors='surrogateescape')
    print('train_o1.py PATCHED -- best_model.pt now only written on improvement')

# hard gate: nothing downstream may run against the buggy saver
assert 'last_model.pt' in TRAIN_PY.read_text(encoding='utf-8', errors='surrogateescape')
for m in [m for m in list(sys.modules) if 'train_o1' in m or 'retrain_long' in m]:
    del sys.modules[m]                        # force reimport of the patched file
print('checkpoint logic verified current')


In [ ]:
# 4) Dependencies (torch/pandas/scipy preinstalled on Colab)
!pip -q install dukascopy-python yfinance pyarrow >/dev/null 2>&1
print('deps ready')

In [ ]:
# 5) Point the data cache + checkpoints at Drive (survive disconnects / resume the pull)
import os, shutil
os.makedirs(f'{PERSIST}/cache/dukascopy', exist_ok=True)
os.makedirs(f'{PERSIST}/cache/long', exist_ok=True)
os.makedirs(f'{PERSIST}/checkpoints_long', exist_ok=True)

def link_to_drive(src, dst):
    """Force src -> dst. The uploaded zip ships real dirs (checkpoints_long,
    data/cache); those must be moved into Drive and replaced by the link, or
    training silently writes to the ephemeral container disk."""
    os.makedirs(os.path.dirname(src) or '.', exist_ok=True)
    if os.path.islink(src):
        if os.path.realpath(src) == os.path.realpath(dst):
            return
        os.unlink(src)
    elif os.path.isdir(src):
        for item in os.listdir(src):
            s, d = os.path.join(src, item), os.path.join(dst, item)
            if not os.path.exists(d):
                shutil.move(s, d)
        shutil.rmtree(src)
    elif os.path.exists(src):
        os.remove(src)
    os.symlink(dst, src)

for src, dst in [('data/cache/dukascopy',        f'{PERSIST}/cache/dukascopy'),
                 ('data/cache/long',             f'{PERSIST}/cache/long'),
                 ('experiments/checkpoints_long', f'{PERSIST}/checkpoints_long')]:
    link_to_drive(src, dst)

# Verify rather than announce. A failed link must stop the notebook here,
# not surface 200 epochs later as a lost run.
for src in ['data/cache/dukascopy', 'data/cache/long', 'experiments/checkpoints_long']:
    assert os.path.islink(src), f'{src} is NOT a symlink -- checkpoints would be lost on disconnect'
    print(f'{src} -> {os.path.realpath(src)}')
print('VERIFIED: cache + checkpoints -> Drive')

## 6) Build the long history for this instrument (cached to Drive)
The Dukascopy pull is the slow part. It caches per-chunk to Drive, so if the session
drops, rerun this cell and it resumes where it left off. Later sessions reuse the cache
and this cell returns almost immediately.

In [ ]:
# 6) Build long data for THIS instrument only (honest per-instrument start)
import sys; sys.path.insert(0, '.')
from data.long_history import build_long
d = build_long(INSTRUMENT)        # start=None -> HONEST_START; end=today
print(INSTRUMENT, len(d['price']), 'H1 bars',
      d['price'].index.min().date(), '->', d['price'].index.max().date())

## 7) Train + evaluate (60 epochs, GPU)
`train_o1` auto-uses CUDA. Runs build_from_frames (regime splits) → train → O3 surprise →
precision/recall, writing checkpoints to Drive and `surprise_timeseries_long_{inst}.json`.

Writes two checkpoints per instrument: **`best_model.pt`** (lowest val — use this one) and
`last_model.pt` (final epoch, kept for inspection only). Before 2026-07-16 these were the
same file and the final epoch silently overwrote the best one.

In [ ]:
# 7) Retrain THIS instrument. Set INSTRUMENT/EPOCHS in cell 2, not here.
import importlib, experiments.retrain_long as R
importlib.reload(R)
R.run(INSTRUMENT, None, None, EPOCHS)   # start=None -> honest per-instrument start

In [ ]:
# 8) Verify this session's outputs, and persist them to Drive
import json, os, glob, shutil, pandas as pd, torch

ck = f'experiments/checkpoints_long/{INSTRUMENT}'
assert os.path.islink('experiments/checkpoints_long'), 'checkpoints dir is NOT linked to Drive!'

hist = json.load(open(f'{ck}/loss_history.json'))
vt   = [v for v in hist['val_total'] if v == v]
best = min(vt); best_ep = vt.index(best) + 1
d    = torch.load(f'{ck}/best_model.pt', map_location='cpu', weights_only=False)
print(f'{INSTRUMENT}: best val={best:.4f} @ epoch {best_ep} | '
      f'best_model.pt holds epoch {d["epoch"]} val={d["val_loss"]:.4f}')
assert abs(float(d['val_loss']) - best) < 1e-6, 'best_model.pt is NOT the best epoch -- do not ship it'

# The val curve plateaus and then wanders: the argmin epoch is partly a draw from
# that noise. Report the plateau, not the lucky minimum.
tail = vt[len(vt)//2:]
if len(tail) > 2:
    import statistics as st
    print(f'   plateau over last {len(tail)} epochs: mean={st.mean(tail):.4f} '
          f'sd={st.stdev(tail):.4f} min={min(tail):.4f} -- quote the mean, not the min')

# retrain_long writes these under ./experiments/, which is NOT the symlinked dir.
# Without this copy they die with the container.
RESULTS = '/content/drive/MyDrive/MWM_run/results'
os.makedirs(RESULTS, exist_ok=True)
for f in sorted(glob.glob('experiments/surprise_timeseries_long_*.json')):
    s_ = json.load(open(f)); ts = pd.to_datetime(s_['timestamps'])
    print(os.path.basename(f), 'test', ts.min().date(), '->', ts.max().date(),
          '| n=', len(s_['surprise_raw']))
    shutil.copy(f, RESULTS)
    print('   -> saved to Drive:', os.path.join(RESULTS, os.path.basename(f)))

print()
print('checkpoints on Drive:', os.path.realpath(ck))
print('results on Drive    :', RESULTS)
print('Done with', INSTRUMENT, '-- set INSTRUMENT to the next one and rerun this notebook.')
